# ML Baseline Evaluation

This notebook trains and evaluates the first baseline models for the Early Warning System.

It uses the feature-engineered ML-ready files:

- `train_feature_engineered_ml_ready.csv`
- `valid_feature_engineered_ml_ready.csv`
- `test_feature_engineered_ml_ready.csv`

The goal is not to build the final best model yet. The goal is to validate whether the target and features contain a learnable signal.

What this notebook does:

1. Load train, validation, and test feature matrices.
2. Check target distribution and class imbalance.
3. Train a Majority-Class baseline.
4. Train a Gaussian Naive Bayes baseline implemented with `numpy`/`pandas`.
5. Evaluate results with:
   - accuracy
   - macro F1
   - weighted F1
   - high-risk recall
   - confusion matrix
   - classification report
6. Calculate feature importance:
   - univariate class-separation F-score
   - top Naive-Bayes class drivers
7. Export all results and recommendations.


## Baseline Method

We compare two baselines:

1. **Majority-Class Baseline**: always predicts the most frequent training label. This tells us what a naive non-ML approach would achieve.
2. **Gaussian Naive Bayes**: a simple probabilistic classifier. It is fast, transparent, and good enough for a first signal check.

For an Early Warning System, the most important metric is usually **High Risk Recall**: how many actually risky product-months we catch before they happen.


In [2]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

TARGET_COL = "next_month_risk_label"
RISK_LABELS = [0, 1, 2]
LABEL_NAMES = {
    0: "Low Risk",
    1: "Medium Risk",
    2: "High Risk"}

INPUT_DIR = Path("../data/feature_engineering")
OUTPUT_DIR = Path("../data/ml_baseline")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
def load_ml_ready_data(input_dir=INPUT_DIR):
    """Load feature-engineered train/validation/test datasets."""
    paths = {
        "train": input_dir / "train_feature_engineered_ml_ready.csv",
        "valid": input_dir / "valid_feature_engineered_ml_ready.csv",
        "test": input_dir / "test_feature_engineered_ml_ready.csv",
    }
    missing = [str(path) for path in paths.values() if not path.exists()]
    if missing:
        raise FileNotFoundError("Missing ML-ready input files:\n" + "\n".join(missing))

    data = {name: pd.read_csv(path) for name, path in paths.items()}
    return data


def split_xy(df):
    """Split features and target."""
    X = df.drop(columns=[TARGET_COL]).copy()
    y = df[TARGET_COL].astype(int).to_numpy()
    return X, y

In [ ]:

def target_distribution(y):
    """Return target class distribution as a dataframe."""
    values, counts = np.unique(y, return_counts=True)
    rows = []
    for label, count in zip(values, counts):
        rows.append({
            "label": int(label),
            "label_name": LABEL_NAMES.get(int(label), str(label)),
            "rows": int(count),
            "share": float(count / len(y))})
    return pd.DataFrame(rows)

def confusion_matrix_np(y_true, y_pred, labels=RISK_LABELS):
    """Compute confusion matrix using numpy only."""
    matrix = np.zeros((len(labels), len(labels)), dtype=int)
    label_to_idx = {label: idx for idx, label in enumerate(labels)}
    for true, pred in zip(y_true, y_pred):
        matrix[label_to_idx[int(true)], label_to_idx[int(pred)]] += 1
    return matrix

In [ ]:
# Compute precision, recall, F1, and support by class
def classification_report_np(y_true, y_pred, labels=RISK_LABELS):
    cm = confusion_matrix_np(y_true, y_pred, labels)
    rows = []
    for i, label in enumerate(labels):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        support = cm[i, :].sum()

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

        rows.append({
            "label": int(label),
            "label_name": LABEL_NAMES.get(int(label), str(label)),
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "support": int(support)})

    report = pd.DataFrame(rows)
    accuracy = np.trace(cm) / cm.sum()
    macro_f1 = report["f1_score"].mean()
    weighted_f1 = np.average(report["f1_score"], weights=report["support"])
    high_risk_recall = report.loc[report["label"] == 2, "recall"].iloc[0]

    summary = {
        "accuracy": float(accuracy),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        "high_risk_recall": float(high_risk_recall)}

    return report, cm, summary

def majority_class_predictor(y_train, n_rows):
    """Predict the most common training class for every row."""
    labels, counts = np.unique(y_train, return_counts=True)
    majority_label = labels[np.argmax(counts)]
    return np.full(n_rows, majority_label, dtype=int), int(majority_label)

In [ ]:
# Simple Gaussian Naive Bayes classifier implemented with numpy
class GaussianNaiveBayes:
    def __init__(self, var_smoothing=1e-6):
        self.var_smoothing = var_smoothing

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=int)
        self.classes_ = np.array(sorted(np.unique(y)))
        self.class_prior_ = {}
        self.theta_ = {}
        self.var_ = {}

        global_var = np.nanvar(X, axis=0)
        epsilon = self.var_smoothing * np.nanmax(global_var)
        if not np.isfinite(epsilon) or epsilon <= 0:
            epsilon = self.var_smoothing
        self.epsilon_ = epsilon

        for cls in self.classes_:
            X_cls = X[y == cls]
            self.class_prior_[cls] = X_cls.shape[0] / X.shape[0]
            self.theta_[cls] = np.nanmean(X_cls, axis=0)
            self.var_[cls] = np.nanvar(X_cls, axis=0) + epsilon

        return self

    def predict_log_proba(self, X):
        X = np.asarray(X, dtype=float)
        log_probs = []
        for cls in self.classes_:
            mean = self.theta_[cls]
            var = self.var_[cls]
            log_prior = np.log(self.class_prior_[cls])
            log_likelihood = -0.5 * np.sum(np.log(2 * np.pi * var))
            log_likelihood -= 0.5 * np.sum(((X - mean) ** 2) / var, axis=1)
            log_probs.append(log_prior + log_likelihood)
        return np.vstack(log_probs).T

    def predict_proba(self, X):
        log_probs = self.predict_log_proba(X)
        shifted = log_probs - log_probs.max(axis=1, keepdims=True)
        probs = np.exp(shifted)
        probs = probs / probs.sum(axis=1, keepdims=True)
        return probs

    def predict(self, X):
        probs = self.predict_proba(X)
        return self.classes_[np.argmax(probs, axis=1)]

In [ ]:
# Create and save evaluation tables for one model
def evaluate_model(model_name, y_true, y_pred):
    report, cm, summary = classification_report_np(y_true, y_pred)
    cm_df = pd.DataFrame(
        cm,
        index=[f"actual_{LABEL_NAMES[label]}" for label in RISK_LABELS],
        columns=[f"pred_{LABEL_NAMES[label]}" for label in RISK_LABELS])

    report.to_csv(OUTPUT_DIR / f"{model_name}_classification_report.csv", index=False)
    cm_df.to_csv(OUTPUT_DIR / f"{model_name}_confusion_matrix.csv")

    with open(OUTPUT_DIR / f"{model_name}_summary_metrics.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    return report, cm_df, summary

In [ ]:
# Calculate an ANOVA-like F-score for class separation
def calculate_univariate_feature_importance(X, y):
    X_values = X.to_numpy(dtype=float)
    y = np.asarray(y, dtype=int)
    features = X.columns.to_numpy()

    overall_mean = np.nanmean(X_values, axis=0)
    classes = np.array(sorted(np.unique(y)))
    n_total = len(y)
    k = len(classes)

    between = np.zeros(X_values.shape[1])
    within = np.zeros(X_values.shape[1])

    for cls in classes:
        X_cls = X_values[y == cls]
        n_cls = X_cls.shape[0]
        cls_mean = np.nanmean(X_cls, axis=0)
        between += n_cls * (cls_mean - overall_mean) ** 2
        within += np.nansum((X_cls - cls_mean) ** 2, axis=0)

    between_df = max(k - 1, 1)
    within_df = max(n_total - k, 1)
    f_score = (between / between_df) / ((within / within_df) + 1e-12)

    importance = pd.DataFrame({
        "feature": features,
        "f_score": f_score,
    }).replace([np.inf, -np.inf], np.nan)
    importance["f_score"] = importance["f_score"].fillna(0)
    importance = importance.sort_values("f_score", ascending=False).reset_index(drop=True)
    return importance


def naive_bayes_class_drivers(model, feature_names, top_n=20):
    """Show features whose class means differ most from overall means."""
    class_means = pd.DataFrame(
        {f"class_{cls}_{LABEL_NAMES[int(cls)]}": model.theta_[cls] for cls in model.classes_},
        index=feature_names,
    )
    class_means["max_abs_class_mean"] = class_means.abs().max(axis=1)
    class_means = class_means.sort_values("max_abs_class_mean", ascending=False)
    return class_means.head(top_n).reset_index(names="feature")

In [ ]:
# Generate practical recommendations based on baseline metrics
def recommendations_from_results(summary_table, target_dist):
    recs = []
    nb_valid = summary_table[
        (summary_table["model"] == "gaussian_naive_bayes")
        & (summary_table["split"] == "valid")
    ].iloc[0]
    majority_valid = summary_table[
        (summary_table["model"] == "majority_class")
        & (summary_table["split"] == "valid")
    ].iloc[0]

    if nb_valid["macro_f1"] <= majority_valid["macro_f1"] + 0.03:
        recs.append({
            "area": "Model signal",
            "recommendation": "The first ML baseline barely improves over the majority baseline. Recheck target rules and add stronger lagged or business-context features.",
        })
    else:
        recs.append({
            "area": "Model signal",
            "recommendation": "The ML baseline improves over the majority baseline, so the feature set contains useful predictive signal.",
        })

    if nb_valid["high_risk_recall"] < 0.60:
        recs.append({
            "area": "High-risk detection",
            "recommendation": "High-risk recall is below 60%. For an Early Warning System, tune the decision threshold or use class-weighted/tree-based models next.",
        })
    else:
        recs.append({
            "area": "High-risk detection",
            "recommendation": "High-risk recall is acceptable for a first baseline. Next, monitor precision so the alert volume stays usable.",
        })

    high_risk_share = target_dist.loc[target_dist["label"] == 2, "share"]
    if len(high_risk_share) and high_risk_share.iloc[0] > 0.35:
        recs.append({
            "area": "Target definition",
            "recommendation": "High-risk class share is quite large. Consider tightening the high-risk target thresholds if business users expect fewer urgent alerts.",
        })
    elif len(high_risk_share) and high_risk_share.iloc[0] < 0.05:
        recs.append({
            "area": "Target definition",
            "recommendation": "High-risk class is rare. Use recall-focused metrics, class weighting, and possibly merge or rebalance labels.",
        })
    else:
        recs.append({
            "area": "Target definition",
            "recommendation": "Target distribution is workable for a first multiclass model. Keep validating thresholds with business assumptions.",
        })

    recs.append({
        "area": "Next model",
        "recommendation": "Next step should be a tree-based model such as Random Forest, Gradient Boosting, XGBoost, or LightGBM when the environment has the needed libraries.",
    })

    return pd.DataFrame(recs)

In [11]:
def run_baseline_pipeline():
    print("Loading feature-engineered ML-ready files...")
    data = load_ml_ready_data()

    X_train, y_train = split_xy(data["train"])
    X_valid, y_valid = split_xy(data["valid"])
    X_test, y_test = split_xy(data["test"])

    print("Rows and features")
    print("-" * 80)
    print(f"Train: X={X_train.shape}, y={y_train.shape}")
    print(f"Valid: X={X_valid.shape}, y={y_valid.shape}")
    print(f"Test:  X={X_test.shape}, y={y_test.shape}")

    train_dist = target_distribution(y_train)
    valid_dist = target_distribution(y_valid)
    test_dist = target_distribution(y_test)
    train_dist.to_csv(OUTPUT_DIR / "train_target_distribution.csv", index=False)
    valid_dist.to_csv(OUTPUT_DIR / "valid_target_distribution.csv", index=False)
    test_dist.to_csv(OUTPUT_DIR / "test_target_distribution.csv", index=False)

    print("\nTraining Majority-Class baseline...")
    majority_valid_pred, majority_label = majority_class_predictor(y_train, len(y_valid))
    majority_test_pred = np.full(len(y_test), majority_label, dtype=int)

    majority_valid_report, majority_valid_cm, majority_valid_summary = evaluate_model(
        "majority_valid", y_valid, majority_valid_pred)
    majority_test_report, majority_test_cm, majority_test_summary = evaluate_model(
        "majority_test", y_test, majority_test_pred)

    print("\nTraining Gaussian Naive Bayes baseline...")
    nb = GaussianNaiveBayes(var_smoothing=1e-4)
    nb.fit(X_train, y_train)
    nb_valid_pred = nb.predict(X_valid)
    nb_test_pred = nb.predict(X_test)

    nb_valid_report, nb_valid_cm, nb_valid_summary = evaluate_model(
        "gaussian_naive_bayes_valid", y_valid, nb_valid_pred)
    nb_test_report, nb_test_cm, nb_test_summary = evaluate_model(
        "gaussian_naive_bayes_test", y_test, nb_test_pred)

    summary_table = pd.DataFrame([
        {"model": "majority_class", "split": "valid", **majority_valid_summary},
        {"model": "majority_class", "split": "test", **majority_test_summary},
        {"model": "gaussian_naive_bayes", "split": "valid", **nb_valid_summary},
        {"model": "gaussian_naive_bayes", "split": "test", **nb_test_summary}])
    summary_table.to_csv(OUTPUT_DIR / "baseline_model_results.csv", index=False)

    print("\nCalculating feature importance...")
    f_importance = calculate_univariate_feature_importance(X_train, y_train)
    f_importance.to_csv(OUTPUT_DIR / "feature_importance_univariate_f_score.csv", index=False)

    nb_drivers = naive_bayes_class_drivers(nb, X_train.columns, top_n=30)
    nb_drivers.to_csv(OUTPUT_DIR / "feature_importance_naive_bayes_class_drivers.csv", index=False)

    recommendations = recommendations_from_results(summary_table, train_dist)
    recommendations.to_csv(OUTPUT_DIR / "baseline_recommendations.csv", index=False)

    report = {
        "train_shape": list(X_train.shape),
        "valid_shape": list(X_valid.shape),
        "test_shape": list(X_test.shape),
        "majority_class_label": int(majority_label),
        "summary_metrics": summary_table.to_dict(orient="records"),
        "top_15_features_univariate_f_score": f_importance.head(15).to_dict(orient="records"),
        "recommendations": recommendations.to_dict(orient="records")}

    with open(OUTPUT_DIR / "baseline_evaluation_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, default=str)

    print("\nBaseline model results")
    print("-" * 80)
    print(summary_table.round(4).to_string(index=False))

    print("\nGaussian Naive Bayes validation confusion matrix")
    print("-" * 80)
    print(nb_valid_cm.to_string())

    print("\nGaussian Naive Bayes validation classification report")
    print("-" * 80)
    print(nb_valid_report.round(4).to_string(index=False))

    print("\nTop 15 feature importance: univariate F-score")
    print("-" * 80)
    print(f_importance.head(15).round(4).to_string(index=False))

    print("\nRecommendations")
    print("-" * 80)
    print(recommendations.to_string(index=False))

    print("\nOutput files saved to:", OUTPUT_DIR.resolve())

    return {
        "summary_table": summary_table,
        "nb_valid_confusion_matrix": nb_valid_cm,
        "nb_valid_classification_report": nb_valid_report,
        "feature_importance": f_importance,
        "recommendations": recommendations}

artifacts = run_baseline_pipeline()


Loading feature-engineered ML-ready files...
Rows and features
--------------------------------------------------------------------------------
Train: X=(62520, 164), y=(62520,)
Valid: X=(18000, 164), y=(18000,)
Test:  X=(16000, 164), y=(16000,)

Training Majority-Class baseline...

Training Gaussian Naive Bayes baseline...

Calculating feature importance...

Baseline model results
--------------------------------------------------------------------------------
               model split  accuracy  macro_f1  weighted_f1  high_risk_recall
      majority_class valid    0.5329    0.2318       0.3705            0.0000
      majority_class  test    0.5339    0.2320       0.3716            0.0000
gaussian_naive_bayes valid    0.4346    0.4284       0.4638            0.4921
gaussian_naive_bayes  test    0.4479    0.4421       0.4754            0.5037

Gaussian Naive Bayes validation confusion matrix
--------------------------------------------------------------------------------
             